# 00 — Setup and environment check

**Run this notebook first, top to bottom.** It does not compute any
science — it just checks that your machine is ready so that notebooks
01–04 can't fail for a silly reason (wrong Python, a missing data file,
an edited constant).

Each cell prints a short report. **Green / "OK" / "PASSED" means good.**
If a cell raises an `AssertionError`, read its message — it tells you
exactly which check failed and how to fix it (usually: install a package,
or run `scripts/fetch_diviner.py`). When the last cell prints
`ALL CHECKS PASSED`, move on to `01_methods.ipynb`.

What gets verified:
1. Python version and the four core libraries
2. The `lunar` package imports, and its physical constants match the literature
3. The bundled Apollo heat-flow data are present and have the expected sensor counts
4. The (optional) Diviner tiles are present
5. SPICE planetary-position kernels load
6. The `output/` folder exists

## 1. Python and package versions

Confirms you are on Python 3.10+ and prints the versions of NumPy, SciPy,
Matplotlib and Pandas. A version mismatch here is the single most common
reason a notebook won't run, so we check it before anything else.

In [ ]:
import sys, platform
print(f'Python {sys.version.split()[0]} on {platform.system()} {platform.machine()}')
assert sys.version_info >= (3, 10), 'Python 3.10+ required'

import numpy, scipy, matplotlib, pandas
for mod in (numpy, scipy, matplotlib, pandas):
    print(f'  {mod.__name__:<12} {mod.__version__}')

## 2. The `lunar` package + its physical constants

This imports `lunar.constants` and **asserts** that the key physical
numbers match the published literature exactly — the Stefan–Boltzmann
constant (CODATA) and the Hayne (2017) regolith parameters
(K_s, K_d, H). These are "hard rules": the whole study is invalid if they
drift, so the code refuses to run if any has been changed by accident.
If an assert fires, someone edited a constant in `lunar/constants.py`.

In [ ]:
import math
from lunar import constants as c

# Hard rules from CLAUDE.md
assert math.isclose(c.SIGMA_SB, 5.670374419e-8, rel_tol=1e-6), 'SIGMA_SB constant wrong'
assert math.isclose(c.K_DEEP, 3.4e-3, rel_tol=1e-6), 'Hayne K_d default is wrong'
assert math.isclose(c.K_SURFACE, 7.4e-4, rel_tol=1e-6), 'Hayne K_s default is wrong'
assert math.isclose(c.H_PARAMETER, 0.06, rel_tol=1e-6), 'Hayne H is wrong'
print('All hard-rule constants verified.')
print(f'  sigma  = {c.SIGMA_SB} W m^-2 K^-4')
print(f'  K_d    = {c.K_DEEP*1e3:.2f} mW m^-1 K^-1 (Hayne 2017)')
print(f'  K_s    = {c.K_SURFACE*1e3:.3f} mW m^-1 K^-1 (Hayne 2017)')
print(f'  H      = {c.H_PARAMETER*100:.1f} cm')

## 3. Apollo heat-flow data integrity

The temperature record we fit is the restored 1971–1977 Apollo Heat-Flow
Experiment (HFE) data set (Nagihara et al. 2018), read by
`lunar.apollo_helpers`. 

The top ~80 cm of each borehole is disturbed by heat conducted down the
metal **borestem**, so we only trust sensors *below* 80 cm. This cell
verifies that the number of usable deep sensors is exactly **7 at Apollo
15** and **16 at Apollo 17**, matching the paper. A mismatch means the
data file or the depth cut changed.

In [ ]:
from lunar.apollo_helpers import extract_sensor_stability
for mission, expected in [('a15', 7), ('a17', 16)]:
    obs = extract_sensor_stability(mission, min_depth_cm=80)
    n_deep = (obs['depth_cm_all'] >= 80).sum()
    print(f'  Apollo {mission[-2:]}: {n_deep} sensors below 80 cm (expected {expected})')
    assert n_deep == expected, f'Sensor count mismatch for {mission}'

## 4. Diviner data (optional)

Diviner is the orbital infrared radiometer on LRO; its "global cumulative"
(GCP) tiles give the *surface* temperature diurnal curve at each latitude.
We use them only for the independent surface-temperature closure check
(Fig 9). **This is optional** — the K_d retrieval itself does not need
Diviner. If the tiles are missing, this cell just tells you to run
`python scripts/fetch_diviner.py` (~310 MB, one time).

In [ ]:
import pathlib
gcp = pathlib.Path('../data/diviner/gcp')
needed = ['global_cumul_avg_cyl_20n30n_002.tab',
          'global_cumul_avg_cyl_10n20n_002.tab']
missing = [f for f in needed if not (gcp / f).exists()]
if missing:
    print('Diviner tiles MISSING — run from project root:')
    print('    python scripts/fetch_diviner.py')
    for f in missing:
        print(f'  [missing] {f}')
else:
    for f in needed:
        size_mb = (gcp / f).stat().st_size // (1<<20)
        print(f'  [ok] {f} ({size_mb} MB)')

## 5. SPICE solar geometry

SPICE is NASA's toolkit for precise planetary positions. The solver needs
to know where the Sun is over a lunation to compute the sunlight hitting
the surface. This cell just confirms the JPL DE440 kernels download and
load, and that a basic Sun-direction call returns a unit vector. The
kernels (~10 MB) are fetched once and cached.

In [ ]:
from lunar.ephem import et_from_iso, solar_vector_moon_frame
import numpy as np
# Verify SPICE kernels load and a basic geometry call works.
et = et_from_iso(['2024-01-01T00:00:00'])
sv = solar_vector_moon_frame(et)
print(f'  solar unit vector at 2024-01-01 (Moon frame): {sv[0]}')
assert abs(np.linalg.norm(sv[0]) - 1.0) < 1e-6, 'should be unit vector'
print('  SPICE kernels OK')

## 6. Output directory ready

Creates `output/` and `output/figures/` if they don't exist — this is
where the pipeline writes its result JSONs and supplementary figures. The
final banner confirms the environment is fully ready.

In [ ]:
import pathlib
out = pathlib.Path('../output')
out.mkdir(exist_ok=True)
(out / 'figures').mkdir(exist_ok=True)
print('  output/ is ready at', out.resolve())

print()
print('================== ALL CHECKS PASSED ==================')
print('Proceed to 01_methods.ipynb.')